# Task 3: Symbolic Recovery

Extract the learned dynamics from the trained Neural ODE and recover the SIR equations using:
- SINDy (sparse regression with physics-motivated library)
- PySR (genetic programming symbolic regression)
- Finite-difference baseline

In [2]:
%cd ..

/Users/invi/Desktop/TopoReformer/Conference/GsOC/sir-model


In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.models.neural_ode import NeuralODE
from src.symbolic.recovery import extract_derivatives, run_sindy, run_pysr
from src.simulation.dataset import load_dataset, train_val_test_split

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cpu


In [4]:
model = NeuralODE(hidden_dim=64, n_layers=3)
model.load_state_dict(torch.load('checkpoints/neural_ode.pt', map_location=device))
model.to(device)
model.eval()
print('Loaded trained Neural ODE')

Loaded trained Neural ODE


## 1. Extract Derivatives from Neural ODE

Sample 10,000 state-parameter points uniformly and evaluate the learned f(s,i,r,beta,gamma) via autograd.

In [5]:
states, params, derivs = extract_derivatives(model, n_points=10000, device=device)

print(f'Extracted derivatives at {len(states)} points')
print(f'States range: s=[{states[:,0].min():.2f}, {states[:,0].max():.2f}], '
      f'i=[{states[:,1].min():.2f}, {states[:,1].max():.2f}]')
print(f'Params range: beta=[{params[:,0].min():.2f}, {params[:,0].max():.2f}], '
      f'gamma=[{params[:,1].min():.2f}, {params[:,1].max():.2f}]')
print(f'\nDerivative statistics:')
for j, name in enumerate(['ds/dt', 'di/dt', 'dr/dt']):
    print(f'  {name}: mean={derivs[:,j].mean():.4f}, std={derivs[:,j].std():.4f}')

Extracted derivatives at 7522 points
States range: s=[0.00, 0.99], i=[0.00, 0.50]
Params range: beta=[0.10, 1.00], gamma=[0.05, 0.50]

Derivative statistics:
  ds/dt: mean=-0.0170, std=0.0308
  di/dt: mean=-0.0107, std=0.0317
  dr/dt: mean=0.0425, std=0.0283


## 2. SINDy: Sparse Identification of Nonlinear Dynamics

Physics-motivated candidate library. Expect to recover:
- ds/dt = -beta * s * i
- di/dt = +beta * s * i - gamma * i
- dr/dt = +gamma * i

In [6]:
sindy_results = run_sindy(states, params, derivs, threshold=0.05)

print('SINDy Recovered Equations:')
print('=' * 60)
for name, res in sindy_results.items():
    print(f'\n{name}:')
    print(f'  {res["equation"]}')
    print(f'  ({res["nonzero_terms"]} nonzero terms)')

print('\n' + '=' * 60)
print('\nGround truth:')
print('  ds/dt = -1.0 * beta*s*i')
print('  di/dt = +1.0 * beta*s*i - 1.0 * gamma*i')
print('  dr/dt = +1.0 * gamma*i')

SINDy Recovered Equations:

ds/dt:
  -6894.1963 * 1 +10547.5037 * s +12209.9769 * i +13840.6333 * r -8969.2278 * s*i -10599.8600 * s*r -12262.1764 * i*r -3653.2522 * s^2 -5315.6385 * i^2 -6946.4489 * r^2 -0.0838 * beta*s -0.1538 * beta*i +0.1940 * gamma*r
  (13 nonzero terms)

di/dt:
  +6941.0442 * 1 -17728.7977 * s -1266.1095 * i -7013.6686 * r +5112.8968 * s*i +10860.5352 * s*r -5602.4683 * i*r +10787.7461 * s^2 -5674.9580 * i^2 +72.6230 * r^2 -0.3379 * gamma*i +0.1642 * beta*i -0.1576 * gamma*r
  (13 nonzero terms)

dr/dt:
  -11148.6948 * 1 +18533.3218 * s -4822.3556 * i +18535.0515 * r +8586.6449 * s*i -14770.9433 * s*r +8584.8223 * i*r -7384.6426 * s^2 +15970.9303 * i^2 -7386.3689 * r^2 +0.4155 * gamma*i
  (11 nonzero terms)


Ground truth:
  ds/dt = -1.0 * beta*s*i
  di/dt = +1.0 * beta*s*i - 1.0 * gamma*i
  dr/dt = +1.0 * gamma*i


In [7]:
# coefficient accuracy
print('Coefficient comparison (nonzero terms only):')
print()

ground_truth = {
    'ds/dt': {'beta*s*i': -1.0},
    'di/dt': {'beta*s*i': 1.0, 'gamma*i': -1.0},
    'dr/dt': {'gamma*i': 1.0},
}

for name in ['ds/dt', 'di/dt', 'dr/dt']:
    print(f'{name}:')
    coeffs = sindy_results[name]['coefficients']
    gt = ground_truth[name]
    for term, true_val in gt.items():
        pred_val = coeffs.get(term, 0.0)
        err = abs(pred_val - true_val)
        print(f'  {term}: predicted={pred_val:+.4f}, true={true_val:+.1f}, error={err:.4f}')
    print()

Coefficient comparison (nonzero terms only):

ds/dt:
  beta*s*i: predicted=+0.0000, true=-1.0, error=1.0000

di/dt:
  beta*s*i: predicted=+0.0000, true=+1.0, error=1.0000
  gamma*i: predicted=-0.3379, true=-1.0, error=0.6621

dr/dt:
  gamma*i: predicted=+0.4155, true=+1.0, error=0.5845



## 3. PySR: Symbolic Regression

Genetic programming search over symbolic expressions. No pre-specified library needed.

In [8]:
# PySR for ds/dt
print('Running PySR for ds/dt...')
pysr_ds = run_pysr(states, params, derivs, compartment='ds/dt', niterations=50)
print('\nBest equation for ds/dt:')
print(pysr_ds)

Running PySR for ds/dt...


/Users/invi/Desktop/TopoReformer/eigen/.venv/lib/python3.12/site-packages/juliacall/__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  warnings.warn(


[juliapkg] Found dependencies: /Users/invi/Desktop/TopoReformer/eigen/.venv/lib/python3.12/site-packages/pysr/juliapkg.json
[juliapkg] Found dependencies: /Users/invi/Desktop/TopoReformer/eigen/.venv/lib/python3.12/site-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /Users/invi/Desktop/TopoReformer/eigen/.venv/lib/python3.12/site-packages/juliapkg/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] WARNING: About to install Julia 1.11.9 to /Users/invi/Desktop/TopoReformer/eigen/.venv/julia_env/pyjuliapkg/install.
[juliapkg]   If you use juliapkg in more than one environment, you are likely to
[juliapkg]   have Julia installed in multiple locations. It is recommended to
[juliapkg]   install JuliaUp (https://github.com/JuliaLang/juliaup) or Julia
[juliapkg]   (https://julialang.org/downloads) yourself.
[juliapkg] Downloading Julia from https://julialang-s3.julialang.

  Installing known registries into `~/.julia`
       Added `General` registry to ~/.julia/registries
    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed IrrationalConstants ───────── v0.2.6
   Installed pixi_jll ──────────────────── v0.41.3+0
   Installed MicroMamba ────────────────── v0.1.15
   Installed ScientificTypesBase ───────── v3.1.0
   Installed DiffRules ─────────────────── v1.15.1
   Installed Tricks ────────────────────── v0.1.13
   Installed Scratch ───────────────────── v1.3.0
   Installed DynamicExpressions ────────── v1.10.4
   Installed Adapt ─────────────────────── v4.5.0
   Installed JSON ──────────────────────── v1.4.0
   Installed OpenSSL_jll ───────────────── v3.0.16+0
   Installed PtrArrays ─────────────────── v1.4.0
   Installed MLJModelInterface ─────────── v1.11.1
   Installed Preferences ───────────────── v1.5.2
   Installed TableTraits ───────────────── v1.0.1
   Installed DiffResults ───────────────── v1

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/Users/invi/Desktop/TopoReformer/eigen/.venv/lib/python3.12/site-packages/pysr/sr.py:1046: FutureWarning: `variable_names` is a data-dependent parameter and should be passed when fit is called. Ignoring parameter; please pass `variable_names` during the call to fit instead.
  warnings.warn(
/Users/invi/Desktop/TopoReformer/eigen/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
[ Info: Started!



Expressions evaluated per second: 2.590e+04
Progress: 145 / 1500 total iterations (9.667%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           9.468e-04  0.000e+00  y = -0.017021
3           6.935e-04  1.557e-01  y = x₃ * -0.038168
5           5.016e-04  1.619e-01  y = (x₃ * -0.097417) * x₀
7           2.942e-04  2.667e-01  y = ((x₂ * 0.88458) - x₃) * 0.07766
9           2.877e-04  1.125e-02  y = (((x₂ * 1.0536) - x₃) * 0.071051) - 0.0070941
11          2.827e-04  8.804e-03  y = ((x₄ - x₃) * ((x₂ + 0.56014) * x₁)) * x₀
13          2.322e-04  9.839e-02  y = ((x₂ * (x₄ + x₄)) - (x₃ * 0.84566)) * (0.08362 - -0.00...
                                      47913)
───────────────────────────────────────────────────────────────────────────────────────────────────
═════════════════════════════

[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           9.468e-04  0.000e+00  y = -0.017022
3           6.935e-04  1.557e-01  y = x₃ * -0.038165
5           3.109e-04  4.011e-01  y = (x₂ - x₃) * 0.078044
7           2.883e-04  3.775e-02  y = ((x₂ - x₃) + -0.072233) * 0.07244
9           2.467e-04  7.801e-02  y = (((x₄ + x₃) * x₂) - x₃) * 0.078044
11          2.280e-04  3.927e-02  y = ((x₃ - ((x₄ + x₂) * x₂)) * 0.078044) / -0.9564
13          2.118e-04  3.697e-02  y = (((x₂ * (x₄ + x₄)) - (x₃ * 0.84566)) * 0.08362) - -0.0...
                                      047913
15          1.874e-04  6.121e-02  y = (x₂ * 0.069196) + (x₃ - (x₃ * (((x₄ * 0.069092) + -0.4...
                                      7918) * -2.3425)))
───────────────────────────────────────────────────────────────────────────────────────────────────

Best equation for ds/dt:
PySRRegressor.equations_ = [
	   pick     score

In [9]:
# PySR for di/dt
print('Running PySR for di/dt...')
pysr_di = run_pysr(states, params, derivs, compartment='di/dt', niterations=50)
print('\nBest equation for di/dt:')
print(pysr_di)

Running PySR for di/dt...


/Users/invi/Desktop/TopoReformer/eigen/.venv/lib/python3.12/site-packages/pysr/sr.py:1046: FutureWarning: `variable_names` is a data-dependent parameter and should be passed when fit is called. Ignoring parameter; please pass `variable_names` during the call to fit instead.
  warnings.warn(
/Users/invi/Desktop/TopoReformer/eigen/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 2.230e+04
Progress: 118 / 1500 total iterations (7.867%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.006e-03  0.000e+00  y = -0.010665
3           8.224e-04  1.008e-01  y = x₄ * -0.056872
5           6.267e-04  1.359e-01  y = (x₄ * -0.16154) * x₂
7           4.401e-04  1.767e-01  y = (x₄ * (x₀ + -0.59265)) / 3.6268
9           2.263e-04  3.327e-01  y = (x₄ + ((x₃ / -1.1046) * x₀)) * -0.13518
11          2.263e-04  2.980e-08  y = (((x₃ * -0.69015) / -5.639) * x₀) + (x₄ * -0.13518)
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions 

[ Info: Final population:
[ Info: Results saved to:



Best equation for di/dt:
PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                       -0.010665413   
	1        0.100791                                  x4 * -0.056872036   
	2        0.135877                            (x4 * -0.16153774) * x2   
	3        0.231244                              ((x0 - x4) * x2) * x1   
	4        0.278125      (x4 + ((x3 / -1.1046034) * x0)) * -0.13517793   
	5        0.000294  (((x3 * x0) * 0.12143941) + (x4 * -0.13744113)...   
	6  >>>>  0.269843  ((x3 + (x4 / (-0.20051993 - (x0 * x0)))) * -0....   
	7        0.031439  ((1.062847 * x3) + (x4 / (-0.20051993 - (x0 * ...   
	
	       loss  complexity  
	0  0.001006           1  
	1  0.000822           3  
	2  0.000627           5  
	3  0.000395           7  
	4  0.000226           9  
	5  0.000226          11  
	6  0.000132          13  
	7  0.000124          15  
]
  - /var/folders/8_/g5skmwpj36d4g7b3fcvrj

In [10]:
# PySR for dr/dt
print('Running PySR for dr/dt...')
pysr_dr = run_pysr(states, params, derivs, compartment='dr/dt', niterations=50)
print('\nBest equation for dr/dt:')
print(pysr_dr)

Running PySR for dr/dt...


/Users/invi/Desktop/TopoReformer/eigen/.venv/lib/python3.12/site-packages/pysr/sr.py:1046: FutureWarning: `variable_names` is a data-dependent parameter and should be passed when fit is called. Ignoring parameter; please pass `variable_names` during the call to fit instead.
  warnings.warn(
/Users/invi/Desktop/TopoReformer/eigen/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 1.850e+04
Progress: 107 / 1500 total iterations (7.133%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           7.986e-04  0.000e+00  y = 0.04248
3           3.441e-04  4.210e-01  y = x₁ * 0.17892
5           1.949e-04  2.843e-01  y = (x₄ * 0.6057) * x₁
7           1.324e-04  1.934e-01  y = (x₄ * (x₁ * 0.49119)) - -0.01219
11          1.210e-04  2.248e-02  y = ((((x₄ * 1.0416) - -0.11873) * x₁) * 0.3747) - -0.0082...
                                      341
15          9.827e-05  5.197e-02  y = (x₁ * 0.4068) / ((2.0973 - ((x₁ * -1.5477) * (x₁ / x₄)...
                                      )) - x₃)
───────────────────────────────────────────────────────────────────────────────────────────────────
══════════════════════════════════════════════════════════

[ Info: Final population:
[ Info: Results saved to:



Best equation for dr/dt:
PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                        0.042480353   
	1        0.421000                                    x1 * 0.17891575   
	2        0.284304                              (x1 * x4) * 0.6056997   
	3        0.193407             (x4 * (x1 * 0.4911855)) - -0.012189695   
	4        0.053308  (x4 - -0.13547158) * ((x1 * 0.37460643) + 0.01...   
	5        0.146839  ((((x3 * 0.31540382) + 0.31705093) * x1) * x4)...   
	6  >>>>  0.185160  (x1 * 0.47069553) / ((1.8645382 - ((x1 / x4) *...   
	7        0.089704  ((x3 + x0) + (1.2022381 - x4)) * (((x4 * x1) -...   
	
	       loss  complexity  
	0  0.000799           1  
	1  0.000344           3  
	2  0.000195           5  
	3  0.000132           7  
	4  0.000119           9  
	5  0.000089          11  
	6  0.000061          13  
	7  0.000051          15  
]
  - /var/folders/8_/g5skmwpj36d4g7b3fcvrj

## 4. Finite-Difference Baseline

Apply SINDy directly to derivatives estimated from mean trajectories (no Neural ODE). This is noisier but serves as a comparison.

In [11]:
data = load_dataset('data/sir_dataset.npz')
train, val, test = train_val_test_split(data)

t_grid = train['t_grid']
dt = t_grid[1] - t_grid[0]

# central differences on mean trajectories
ds_dt = np.gradient(train['mean_s'], dt, axis=1)
di_dt = np.gradient(train['mean_i'], dt, axis=1)
dr_dt = np.gradient(train['mean_r'], dt, axis=1)

# flatten: each (trajectory, time_point) is one data point
s_flat = train['mean_s'].flatten()
i_flat = train['mean_i'].flatten()
r_flat = (1.0 - train['mean_s'] - train['mean_i']).flatten()

n_traj, n_time = train['mean_s'].shape
beta_flat = np.repeat(train['params'][:, 0], n_time)
gamma_flat = np.repeat(train['params'][:, 1], n_time)

fd_states = np.column_stack([s_flat, i_flat, r_flat])
fd_params = np.column_stack([beta_flat, gamma_flat])
fd_derivs = np.column_stack([ds_dt.flatten(), di_dt.flatten(), dr_dt.flatten()])

# subsample to avoid huge arrays
rng = np.random.default_rng(42)
idx = rng.choice(len(fd_states), size=min(20000, len(fd_states)), replace=False)
fd_states, fd_params, fd_derivs = fd_states[idx], fd_params[idx], fd_derivs[idx]

fd_results = run_sindy(fd_states, fd_params, fd_derivs, threshold=0.05)

print('SINDy on Finite Differences (baseline):')
print('=' * 60)
for name, res in fd_results.items():
    print(f'\n{name}:')
    print(f'  {res["equation"]}')
    print(f'  ({res["nonzero_terms"]} nonzero terms)')

SINDy on Finite Differences (baseline):

ds/dt:
  -0.8385 * beta*s*i
  (1 nonzero terms)

di/dt:
  +0.8400 * beta*s*i -1.0025 * gamma*i
  (2 nonzero terms)

dr/dt:
  +0.9996 * gamma*i
  (1 nonzero terms)


## 5. Comparison

Summary of all recovered equations across methods.

In [12]:
print('Method Comparison')
print('=' * 70)
print(f'{"Method":<25} {"ds/dt":<20} {"di/dt":<25} {"dr/dt":<15}')
print('-' * 70)
print(f'{"Ground Truth":<25} {"-beta*s*i":<20} {"beta*s*i - gamma*i":<25} {"gamma*i":<15}')
print(f'{"SINDy (Neural ODE)":<25} {sindy_results["ds/dt"]["nonzero_terms"]:>3} terms{"":>12} {sindy_results["di/dt"]["nonzero_terms"]:>3} terms{"":>17} {sindy_results["dr/dt"]["nonzero_terms"]:>3} terms')
print(f'{"SINDy (Finite Diff)":<25} {fd_results["ds/dt"]["nonzero_terms"]:>3} terms{"":>12} {fd_results["di/dt"]["nonzero_terms"]:>3} terms{"":>17} {fd_results["dr/dt"]["nonzero_terms"]:>3} terms')
print('\nFull equations:')
for method, res in [('SINDy (Neural ODE)', sindy_results), ('SINDy (Finite Diff)', fd_results)]:
    print(f'\n  {method}:')
    for name in ['ds/dt', 'di/dt', 'dr/dt']:
        print(f'    {name} = {res[name]["equation"]}')

Method Comparison
Method                    ds/dt                di/dt                     dr/dt          
----------------------------------------------------------------------
Ground Truth              -beta*s*i            beta*s*i - gamma*i        gamma*i        
SINDy (Neural ODE)         13 terms              13 terms                   11 terms
SINDy (Finite Diff)         1 terms               2 terms                    1 terms

Full equations:

  SINDy (Neural ODE):
    ds/dt = -6894.1963 * 1 +10547.5037 * s +12209.9769 * i +13840.6333 * r -8969.2278 * s*i -10599.8600 * s*r -12262.1764 * i*r -3653.2522 * s^2 -5315.6385 * i^2 -6946.4489 * r^2 -0.0838 * beta*s -0.1538 * beta*i +0.1940 * gamma*r
    di/dt = +6941.0442 * 1 -17728.7977 * s -1266.1095 * i -7013.6686 * r +5112.8968 * s*i +10860.5352 * s*r -5602.4683 * i*r +10787.7461 * s^2 -5674.9580 * i^2 +72.6230 * r^2 -0.3379 * gamma*i +0.1642 * beta*i -0.1576 * gamma*r
    dr/dt = -11148.6948 * 1 +18533.3218 * s -4822.3556 * i +1853